# Four sampled control methods on five tasks

Q-learning, SARSA, Expected SARSA and Double Q receive sampled transitions. Artificial time limits keep continuation value; true termination suppresses it. Model-aware optimality gaps are computed only after fitting, separately from finite-rollout returns.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src/decision_lab').is_dir()
sys.path.insert(0, str(ROOT / 'src'))
def read(path): return json.loads((ROOT / path).read_text())


## Expected behavior and a time-limit oracle

In [2]:
from decision_lab.tabular import epsilon_distribution, bootstrap_target
p = epsilon_distribution(np.array([1.,3.,999.]),np.array([True,True,False]),.2)
assert np.allclose(p,[.1,.9,0])
assert bootstrap_target(1.,10.,False,.9)==10.
assert bootstrap_target(1.,10.,True,.9)==1.
print("behavior probabilities",p)

behavior probabilities [0.1 0.9 0. ]


## Every fitted seed

In [3]:
fits = pd.read_csv(ROOT / "docs/individual_fits.csv")
tabular = fits[fits.task.isin(["Wireless","FrozenLake-v1","CliffWalking-v1","Taxi-v3","Maze"])]
assert len(tabular)==60
tabular.groupby(["task","method"]).test_return.agg(["mean","std","min"])

mean       std       min
task            method                                        
CliffWalking-v1 double_q        -13.000000  0.000000  -13.0000
                expected_sarsa  -17.000000  0.000000  -17.0000
                q_learning      -13.000000  0.000000  -13.0000
                sarsa           -17.000000  0.000000  -17.0000
FrozenLake-v1   double_q          0.630000  0.111355    0.5100
                expected_sarsa    0.706667  0.041633    0.6600
                q_learning        0.730000  0.000000    0.7300
                sarsa             0.626667  0.100664    0.5200
Maze            double_q          0.815800  0.000000    0.8158
                expected_sarsa    0.815700  0.000173    0.8155
                q_learning        0.815500  0.000000    0.8155
                sarsa             0.815600  0.000173    0.8155
Taxi-v3         double_q          8.020000  0.000000    8.0200
                expected_sarsa    7.336667  1.183568    5.9700
                q_learning        6.656667  2.344063    3.9500
                sarsa             2.556667  1.189384    1.8600
Wireless        double_q       -648.116667  3.036879 -651.0900
                expected_sarsa -647.653333  1.298627 -648.8300
                q_learning     -646.430000  2.035755 -648.6900
                sarsa          -642.266667  4.323024 -645.3200

## Fresh published policy deployment

In [4]:
from decision_lab.deployment import load_policy
from decision_lab.tabular import rollout_policy
record, make, policy = load_policy(ROOT / "assets/policies/tabular_Maze_q_learning_0.json")
evidence = rollout_policy(make,policy,np.arange(3)+300000)
print(evidence["returns"],evidence["lengths"])

[0.83 0.78 0.82] [18 23 19]


Full fits: `decision-lab tabular --seed 0` (repeat seeds 1 and 2 where applicable). See [protocol](../docs/protocol.md), [results](../docs/results.md) and [references](../docs/references.md). These cells are numerical checks and evidence readers, not replacement training budgets.